# NC_ResNet20_MNIST

**Fill the missing (ResNet-20, MNIST) cell in the (Architecture × Dataset) grid.**

| | MNIST | CIFAR-10 |
|---|---|---|
| MLP-5 | 1.052 ✓ | 0.901 ✓ |
| ResNet-20 | **← this notebook** | 1.515 ✓ |

**Protocol:** identical to `NC_CIFAR10_Baseline` — SGD, lr=0.1, wd=1e-3, MultiStepLR, 3 seeds.  
**Key adaptation:** MNIST 28×28 padded to 32×32, grayscale repeated to 3 channels.  
**Expected runtime:** ~15–20 min per seed on T4 (MNIST is ~3× faster than CIFAR-10).


In [11]:

# Change this:
SAVE_DIR = '/kaggle/working/'

# To this (Colab):
SAVE_DIR = '/tmp/'

# Or create the directory first:
import os
SAVE_DIR = '/kaggle/working/'
os.makedirs(SAVE_DIR, exist_ok=True)

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# NC_ResNet20_MNIST.ipynb  —  fill the missing (ResNet-20, MNIST) cell
#
# Question: what is fn* for ResNet-20 trained on MNIST?
# This completes the 2×2 (architecture × dataset) grid and allows us to
# isolate the architecture effect on MNIST (independently of CIFAR-10).
#
# Protocol mirrors NC_CIFAR10_Baseline exactly:
#   - ResNet-20 (same architecture, same init)
#   - SGD, lr=0.1, momentum=0.9, nesterov, wd=1e-3
#   - Phase 1: CE 200 epochs  → ≥99% train accuracy
#   - Phase 2: MSE 600 epochs → NC1 < 0.01
#   - No augmentation (standard for NC experiments)
#   - 3 seeds
#
# Key difference from CIFAR: MNIST is 1-channel 28×28 grayscale
#   → pad to 32×32, repeat channel to 3 (standard MNIST→ResNet adaptation)
#   → batch size 512 (larger than CIFAR 128 since MNIST is faster)
#
# Expected results (informed guess from MLP vs ResNet CIFAR gap):
#   fn* for ResNet/MNIST should be higher than MLP/MNIST (1.052)
#   if architecture is the primary driver, expect fn* ≈ 1.4–1.6
#   if dataset is the primary driver, expect fn* ≈ 1.52 (≈ ResNet/CIFAR)
# ══════════════════════════════════════════════════════════════════════════════


In [2]:
import torch, torchvision, time
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = '/kaggle/working/'   # change to '/tmp/' on Colab
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True

print(f'GPU: {torch.cuda.get_device_name(0)}  |  PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')
print()
print('Known fn* values:')
print('  MNIST    MLP-5:     1.052 ± 0.063  (CV=5.9%,  N=12)')
print('  CIFAR-10 MLP-5:     0.901 ± 0.053  (CV=5.9%,  N=3)')
print('  CIFAR-10 ResNet-20: 1.515 ± 0.007  (CV=0.4%,  N=3)')
print('  MNIST    ResNet-20: ???  ← this experiment')


GPU: NVIDIA A100-SXM4-40GB  |  PyTorch: 2.10.0+cu128
Device: cuda

Known fn* values:
  MNIST    MLP-5:     1.052 ± 0.063  (CV=5.9%,  N=12)
  CIFAR-10 MLP-5:     0.901 ± 0.053  (CV=5.9%,  N=3)
  CIFAR-10 ResNet-20: 1.515 ± 0.007  (CV=0.4%,  N=3)
  MNIST    ResNet-20: ???  ← this experiment


In [3]:
# Padding to 32×32 is needed because ResNet-20 uses stride-2 downsampling.
# No augmentation — required for NC experiments (augmentation inflates
# within-class variability and suppresses NC1).
# Repeat grayscale to 3 channels using Lambda (minimal, no colour conversion).

transform = T.Compose([
    T.Pad(2),                              # 28×28 → 32×32
    T.ToTensor(),
    T.Normalize((0.1307,), (0.3081,)),
    T.Lambda(lambda x: x.repeat(3, 1, 1)), # 1×32×32 → 3×32×32
])

trainset = torchvision.datasets.MNIST(
    root='/tmp/data', train=True,  download=True, transform=transform)
testset  = torchvision.datasets.MNIST(
    root='/tmp/data', train=False, download=True, transform=transform)

train_loader = DataLoader(trainset, batch_size=512, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=1024, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'MNIST: {len(trainset):,} train / {len(testset):,} test')
print(f'Train batches: {len(train_loader)} x 512')


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.06MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 132kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.9MB/s]

MNIST: 60,000 train / 10,000 test
Train batches: 118 x 512


In [4]:
# Exact same architecture as NC_CIFAR10_Baseline — no changes.
# Feature dimension = 64 (output of global average pooling).

class BasicBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_c)
        self.skip  = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.skip = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c))
    def forward(self, x):
        return F.relu(
            self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))
            + self.skip(x))

class ResNet20(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1  = nn.Conv2d(3, 16, 3, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(16)
        self.layer1 = self._make(16, 16, 3, stride=1)
        self.layer2 = self._make(16, 32, 3, stride=2)
        self.layer3 = self._make(32, 64, 3, stride=2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.fc     = nn.Linear(64, num_classes)
        self._feats = None
        self.pool.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.flatten(1).detach()))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make(self, in_c, out_c, n, stride):
        layers = [BasicBlock(in_c, out_c, stride)]
        for _ in range(n - 1):
            layers.append(BasicBlock(out_c, out_c, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        x = self.pool(x)
        return self.fc(x.flatten(1))

    def get_features(self, x):
        self(x); return self._feats

m = ResNet20()
print(f'ResNet-20 (MNIST variant): feats=(batch,64)  params={sum(p.numel() for p in m.parameters())/1e6:.3f}M')
del m


ResNet-20 (MNIST variant): feats=(batch,64)  params=0.272M


In [5]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += len(y)
    return correct / total

@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval()
    feats, labels = [], []
    for x, y in loader:
        feats.append(model.get_features(x.to(DEVICE)).cpu())
        labels.append(y)
    H = torch.cat(feats).float()
    Y = torch.cat(labels)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y == c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    # NC1: within/between covariance ratio
    Sw = sum((H[Y==c]-mu_c[c]).T @ (H[Y==c]-mu_c[c]) for c in range(K)) / len(H)
    Sb = (M.T @ M) / K
    nc1 = (torch.trace(Sw) / torch.trace(Sb)).item()
    # NC2: ETF deviation
    M_n   = F.normalize(M, dim=1)
    gram  = M_n @ M_n.T
    tgt   = (torch.eye(K) - torch.ones(K,K)/K).to(gram) * (K/(K-1))
    nc2   = (gram - tgt).norm().item() / K
    # NC3: self-duality
    W     = F.normalize(model.fc.weight.detach().cpu(), dim=1)
    nc3   = (1 - (W * M_n).sum(1).mean()).item()
    # mean feature norm
    fn    = H.norm(dim=1).mean().item()
    return {'nc1': nc1, 'nc2': nc2, 'nc3': nc3, 'feat_norm': fn}


In [6]:
# SGD + MultiStepLR + wd=1e-3 — same as NC_CIFAR10_Baseline
# Phase-2 milestones: 300, 450 (same as CIFAR)
# MNIST trains ~3× faster than CIFAR-10 per epoch

def run(model, name, lr=0.1, wd=1e-3,
        phase1=200, phase2=600, nc_every=10, nc_thresh=0.01):
    model = model.to(DEVICE)
    K = 10; rows = []; terminal = False; t0 = time.time()

    for phase, loss_name, n_ep, milestones in [
        (1, 'ce',  phase1, [100, 150]),
        (2, 'mse', phase2, [300, 450]),
    ]:
        opt = torch.optim.SGD(model.parameters(), lr=lr,
                              momentum=0.9, weight_decay=wd, nesterov=True)
        sched = torch.optim.lr_scheduler.MultiStepLR(
                    opt, milestones=milestones, gamma=0.1)
        off = phase1 if phase == 2 else 0

        for ep_l in range(1, n_ep + 1):
            ep = off + ep_l
            model.train()
            for x, y in train_loader:
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                if loss_name == 'mse':
                    loss = F.mse_loss(logits, F.one_hot(y, K).float())
                else:
                    loss = F.cross_entropy(logits, y)
                loss.backward(); opt.step()
            sched.step()

            if ep_l % nc_every == 0 or ep_l == n_ep:
                tr = evaluate(model, train_loader)
                te = evaluate(model, test_loader)
                if tr >= 0.99 and not terminal:
                    terminal = True
                    print(f'  [{name}] Terminal phase at epoch {ep}')
                nc_metrics = (compute_nc(model, train_loader)
                              if terminal
                              else {'nc1': None, 'nc2': None,
                                    'nc3': None, 'feat_norm': None})
                rows.append({'epoch': ep, 'phase': phase,
                             'train': tr, 'test': te, **nc_metrics})
                mins = (time.time() - t0) / 60
                nc1_str = f"{nc_metrics['nc1']:.5f}" if nc_metrics['nc1'] else 'N/A'
                fn_str  = f"{nc_metrics['feat_norm']:.4f}" if nc_metrics['feat_norm'] else 'N/A'
                print(f'  ep={ep:>4} tr={tr:.4f} te={te:.4f} '
                      f'nc1={nc1_str} fn={fn_str} t={mins:.1f}m')

                if nc_metrics['nc1'] and nc_metrics['nc1'] < nc_thresh:
                    print(f'  *** T_NC={ep}  fn*={nc_metrics["feat_norm"]:.4f}  (NC1<{nc_thresh})')
                    df = pd.DataFrame(rows)
                    return df, ep, nc_metrics['feat_norm']

    df = pd.DataFrame(rows)
    return df, None, None


In [7]:
results = []
for seed in range(3):
    print(f'\n=== ResNet-20 MNIST seed={seed} ===')
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = ResNet20()
    df, t_nc, fn = run(model, f'ResNet20-MNIST-s{seed}',
                       lr=0.1, wd=1e-3, phase1=200, phase2=600)
    df.to_csv(f'{SAVE_DIR}resnet20_mnist_s{seed}.csv', index=False)
    status = f'T_NC={t_nc}  fn*={fn:.4f}' if t_nc else \
             f'DNF  final_nc1={df.dropna(subset=["nc1"]).nc1.iloc[-1]:.5f}'
    print(f'  => {status}')
    results.append({'seed': seed, 'T_NC': t_nc, 'fn': fn})

summary = pd.DataFrame(results)
summary.to_csv(SAVE_DIR + 'resnet20_mnist_summary.csv', index=False)

print('\n=== RESULTS ===')
print(summary.to_string())
confirmed = summary.dropna(subset=['fn'])
if len(confirmed):
    fns = confirmed.fn.values
    print(f'\nfn* values: {[round(f,4) for f in fns]}')
    print(f'mean = {np.mean(fns):.4f}  std = {np.std(fns):.4f}  CV = {np.std(fns)/np.mean(fns)*100:.1f}%')
    print()
    print('=== GRID UPDATE ===')
    print(f'{"":15s} {"MNIST":>10} {"CIFAR-10":>10}')
    print(f'{"MLP-5":15s} {"1.052":>10} {"0.901":>10}')
    print(f'{"ResNet-20":15s} {np.mean(fns):>10.3f} {"1.515":>10}')
    print()
    arch_mnist = np.mean(fns) - 1.052
    arch_cifar = 1.515 - 0.901
    data_mlp   = 0.901 - 1.052
    data_resnet = 1.515 - np.mean(fns)
    print(f'Architecture effect on MNIST:  {arch_mnist:+.3f} ({arch_mnist/1.052*100:.0f}%)')
    print(f'Architecture effect on CIFAR:  {arch_cifar:+.3f} ({arch_cifar/0.901*100:.0f}%)')
    print(f'Dataset effect on MLP-5:       {data_mlp:+.3f} ({data_mlp/1.052*100:.0f}%)')
    print(f'Dataset effect on ResNet-20:   {data_resnet:+.3f} ({data_resnet/np.mean(fns)*100:.0f}%)')



=== ResNet-20 MNIST seed=0 ===
  ep=  10 tr=0.9858 te=0.9831 nc1=N/A fn=N/A t=1.8m
  ep=  20 tr=0.9387 te=0.9451 nc1=N/A fn=N/A t=3.6m
  ep=  30 tr=0.9745 te=0.9702 nc1=N/A fn=N/A t=5.4m
  ep=  40 tr=0.8685 te=0.8654 nc1=N/A fn=N/A t=7.2m
  ep=  50 tr=0.9660 te=0.9628 nc1=N/A fn=N/A t=9.0m
  [ResNet20-MNIST-s0] Terminal phase at epoch 60
  ep=  60 tr=0.9952 te=0.9914 nc1=0.03636 fn=5.6792 t=10.9m
  ep=  70 tr=0.9901 te=0.9872 nc1=0.04516 fn=5.7118 t=12.8m
  ep=  80 tr=0.9490 te=0.9374 nc1=0.05736 fn=5.2448 t=14.8m
  ep=  90 tr=0.9909 te=0.9869 nc1=0.04236 fn=5.5422 t=16.7m
  ep= 100 tr=0.9962 te=0.9931 nc1=0.03676 fn=5.7868 t=18.6m
  ep= 110 tr=1.0000 te=0.9963 nc1=0.00729 fn=5.8577 t=20.6m
  *** T_NC=110  fn*=5.8577  (NC1<0.01)


OSError: Cannot save file into a non-existent directory: '/kaggle/working'

In [14]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plt.rcParams.update({'font.family': 'serif', 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

colors = ['#2196F3', '#4CAF50', '#F44336']

for seed in range(3):
    try:
        df_s = pd.read_csv(f'{SAVE_DIR}resnet20_mnist_s{seed}.csv')
        nc_s = df_s.dropna(subset=['nc1'])
        c = colors[seed]
        p1 = df_s[df_s.phase == 1]
        p2 = df_s[df_s.phase == 2]

        axes[0].plot(df_s.epoch, df_s.train, color=c, lw=2, label=f's{seed}')
        axes[0].plot(df_s.epoch, df_s.test,  color=c, lw=2, ls='--', alpha=0.7)

        if len(nc_s):
            axes[1].semilogy(nc_s.epoch, nc_s.nc1, color=c, lw=2, label=f's{seed}')
            axes[2].plot(nc_s.epoch, nc_s.feat_norm, color=c, lw=2, label=f's{seed}')
    except Exception as e:
        print(f'Plot error seed {seed}: {e}')

axes[0].axvline(200, color='gray', ls=':', lw=1.5)
axes[0].set(xlabel='Epoch', ylabel='Accuracy', title='(a) Accuracy')
axes[0].legend(fontsize=9)

axes[1].axvline(200, color='gray', ls=':', lw=1.5)
axes[1].axhline(0.01, color='black', ls='--', lw=1.5, label='NC1=0.01')
axes[1].set(xlabel='Epoch', ylabel='NC1 (log)', title='(b) NC1 collapse')
axes[1].legend(fontsize=9)

axes[2].axvline(200, color='gray', ls=':', lw=1.5)
# Mark known fn* values for comparison
axes[2].axhline(1.052, color='#2166ac', ls='--', lw=1.5, alpha=0.6,
                label='MNIST MLP-5 fn*=1.052')
axes[2].axhline(1.515, color='#d6604d', ls='--', lw=1.5, alpha=0.6,
                label='CIFAR ResNet fn*=1.515')
if len(confirmed):
    axes[2].axhline(np.mean(fns), color='green', ls='-', lw=2,
                    label=f'MNIST ResNet fn*={np.mean(fns):.3f}')
axes[2].set(xlabel='Epoch', ylabel='Feature norm (fn)',
            title='(c) Feature norm — vs known fn* values')
axes[2].legend(fontsize=8)

fig.suptitle('ResNet-20 on MNIST — Completing the (Architecture × Dataset) Grid',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig_resnet20_mnist.png', dpi=150, bbox_inches='tight')
print(f'\nSaved: fig_resnet20_mnist.png')
print('Also saving per-seed CSVs: resnet20_mnist_s{0,1,2}.csv')
print('Summary: resnet20_mnist_summary.csv')


Plot error seed 1: [Errno 2] No such file or directory: '/kaggle/working/resnet20_mnist_s1.csv'
Plot error seed 2: [Errno 2] No such file or directory: '/kaggle/working/resnet20_mnist_s2.csv'

Saved: fig_resnet20_mnist.png
Also saving per-seed CSVs: resnet20_mnist_s{0,1,2}.csv
Summary: resnet20_mnist_summary.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')